# 实验 05：完整记忆库中的 Jacobian 秩坍缩——开发预检

主问题是：

> 在第一步竞争程度相同的条件下，softmax 与 sparsemax 保留输入方向的速度是否仍然不同？

这个 Notebook **只执行开发预检**：`1 memory seed × 12 targets × 2 masks`。它验证实现、运行预算、IPR 共同覆盖和类别覆盖；不运行 8 个正式 seed，也不产生 05A 的确认性结论。

本轮输入是 100 条 128 维随机记忆及其三档 Hamming 噪声查询。每一步由全部记忆共同读出；变换结果是 memory span 内的一步 Jacobian $A_t$、累计 Jacobian $J_t$ 和 $G_t=J_t^\top J_t$ 的完整谱。输出包括方向存活率、数值自检、删失/饱和审计与开发诊断图。

## 1. 下载并核对冻结源码

Notebook 固定到开发预检提交 `bf259d6`。核心脚本和 protocol 都核对 SHA-256；任一文件不一致便停止。

In [ ]:
import hashlib
import importlib.util
from pathlib import Path
import subprocess
import sys
import urllib.request

required = {
    "torch": "torch",
    "entmax": "entmax",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

CODE_REV = "bf259d6bf3b2cb2e0102e85f5c09e1759dce5fc8"
BASE = Path("/content/hopfield-dynamic-geometry")
SOURCES = {
    "experiment_05_multimemory_rank_collapse.py": "b79a6c699924ab324eb6d5c9c1794388f6592376d6da9a4a5311f598e147517a",
    "experiment_05_multimemory_rank_collapse_protocol.md": "0c58649718fb6b5173a04f67c00aae01f8339930446410660480132f20ed1bec",
}
BASE.mkdir(parents=True, exist_ok=True)
raw_root = f"https://raw.githubusercontent.com/Heptazero/nn-labs/{CODE_REV}/representation-geometry/experiments/hopfield-dynamic-geometry"
for name, expected in SOURCES.items():
    target = BASE / name
    urllib.request.urlretrieve(f"{raw_root}/{name}", target)
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError(f"SHA-256 mismatch for {name}: {actual}")
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
print("verified source revision:", CODE_REV)

## 2. 冻结条件与判定边界

开发预检依次做四件事：

1. 检查白化图册、自动微分投影、Ritz 上界、正交补清零、解析 Jacobian、PSD 谱和累计递推共 7 项恒等式。
2. 扫描第一步 IPR（参与竞争的有效记忆数），在 softmax 与 sparsemax 的共同范围内选择低、中、高三个对数间隔水平。
3. 只有每档相对匹配误差不超过 5%，才计算 `t=0..12` 的 Jacobian 谱。
4. 审计第一步即坍缩、灵敏度灭绝、阈值删失和非有限值。

`formal_05A_ready=True` 只表示可以进入正式运行。开发曲线即使看起来分开，也不能用于接受或拒绝 05A。

In [ ]:
from experiment_05_multimemory_rank_collapse import (
    RankCollapseConfig,
    run_development_preflight,
)

config = RankCollapseConfig()
output_dir = Path("/content/experiment_05_development")
summary = run_development_preflight(config, output_dir)
print("status:", summary["status"])
print("formal 05A ready:", summary["formal_05A_ready"])
print("confirmatory claim allowed:", summary["confirmatory_claim_allowed"])
print("runtime seconds:", round(summary["development_runtime_seconds"], 2))

## 3. 检查门槛、匹配点和数据审计

先看自检是否全过，再看每个噪声率是否都有三档匹配点。`collapsed_at_entry` 是第一步已经只剩一个方向；`extinct_at_final` 是累计敏感度到第 12 步已低于冻结阈值。两者是可解释状态，不是自动删除的数据。

In [ ]:
import pandas as pd
from IPython.display import display

self_checks = pd.read_csv(output_dir / "self_checks.csv")
ipr_ranges = pd.read_csv(output_dir / "ipr_common_ranges.csv")
matched = pd.read_csv(output_dir / "matched_ipr_levels.csv")
rank_audit = pd.read_csv(output_dir / "rank_data_audit.csv")

display(self_checks)
display(ipr_ranges)
display(matched[[
    "rho", "level", "requested_log_ipr", "target_ipr", "method", "alpha",
    "achieved_median_ipr", "relative_match_error",
]])
display(rank_audit)
display(pd.DataFrame(summary["endpoint_category_counts"]))

assert summary["all_seven_self_checks_passed"]
assert summary["all_rhos_have_three_matched_ipr_levels"]
assert summary["maximum_ipr_match_relative_error"] <= config.ipr_match_relative_tolerance
assert summary["rank_data_audit"]["unexpected_nonfinite_count"] == 0
assert summary["formal_05A_ready"]
assert not summary["confirmatory_claim_allowed"]

## 4. 查看开发诊断图

A 检查两种竞争函数能否覆盖共同 IPR；B、C 只用于确认秩轨迹能被稳定记录；D 检查四类终点是否出现。图题明确标记为 development-only。

In [ ]:
from IPython.display import Image

display(Image(filename=str(output_dir / "development_figure.png")))

## 5. 下一道门

本 Notebook 全部断言通过后，才能另行实现并运行 8 个独立 memory seed 的正式 05A。正式结论必须以 seed 为独立单位，检查配对 `dimension_auc` 差异是否达到 0.10 且 95% bootstrap CI 不跨 0。

如果正式曲线不分，停止“变体坍缩几何”路线；如果差异完全跟随 support、entropy 或 gap，只把秩谱作为稀疏竞争的几何表达；只有匹配后仍有差异并与检索结局相关，才进入条件性的 05B。

## 6. 下载开发预检产物

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive(
    "/content/experiment_05_development_artifacts", "zip", root_dir=output_dir
)
print("archive:", archive)
files.download(archive)